# LeetCode #1117: Building H2O

https://leetcode.com/problems/building-h2o/

## Synchronization Approaches

| Approach | Mechanism | Notes |
| :--- | :--- | :--- |
| **Naive: Busy-Wait** | Spin on shared counters | Wastes CPU; race-prone |
| **Optimal: Barrier + Semaphores ★** | `SemaphoreSlim` for 2 H and 1 O per molecule | Ensures exact 2:1 ratio before any thread passes the barrier |

---

## Understanding the Methods

### Naive: Busy-Wait
Spin in a loop checking a shared flag. Works but burns CPU.

### Optimal: Barrier + Semaphores ★
Use a semaphore with capacity 2 for hydrogen and capacity 1 for oxygen. Both semaphores control entry; a `Barrier(3)` ensures all three threads (2H + 1O) rendezvous before any bonds. After the barrier, reset oxygen's semaphore and release hydrogen's for the next molecule.

**Constraints:**
* Threads call `hydrogen()` and `oxygen()` in arbitrary order
* Must output groups of `HHO` in any internal order
* Total H calls = 2 * total O calls


## Solutions
### C#

In [ ]:
using System.Threading;

public class H2O
{
    private readonly SemaphoreSlim _hSem = new SemaphoreSlim(2, 2);
    private readonly SemaphoreSlim _oSem = new SemaphoreSlim(1, 1);
    private readonly Barrier       _barrier = new Barrier(3);

    public void Hydrogen(Action releaseHydrogen)
    {
        _hSem.Wait();
        _barrier.SignalAndWait();
        releaseHydrogen();
        // last hydrogen to exit resets the H semaphore implicitly via Release calls
        _hSem.Release(); // not needed here; barrier resets state after each molecule
    }

    public void Oxygen(Action releaseOxygen)
    {
        _oSem.Wait();
        _barrier.SignalAndWait();
        releaseOxygen();
        _hSem.Release();
        _hSem.Release();
        _oSem.Release();
    }
}

### Python

In [ ]:
import threading

class H2O:
    def __init__(self):
        self.h_sem   = threading.Semaphore(2)   # 2 hydrogens per molecule
        self.o_sem   = threading.Semaphore(1)   # 1 oxygen per molecule
        self.barrier = threading.Barrier(3)     # wait for HHO to assemble

    def hydrogen(self, releaseHydrogen: 'Callable[[], None]') -> None:
        self.h_sem.acquire()
        self.barrier.wait()
        releaseHydrogen()

    def oxygen(self, releaseOxygen: 'Callable[[], None]') -> None:
        self.o_sem.acquire()
        self.barrier.wait()
        releaseOxygen()
        # release for next molecule
        self.h_sem.release()
        self.h_sem.release()
        self.o_sem.release()

### Go

In [ ]:
package main

import "sync"

type H2O struct {
	hCh chan struct{}   // capacity 2
	oCh chan struct{}   // capacity 1
	wg  sync.WaitGroup // used as barrier alternative
	mu  sync.Mutex
	cnt int
	done chan struct{}
}

// Simple semaphore-based approach using buffered channels
func NewH2O() *H2O {
	h := &H2O{
		hCh:  make(chan struct{}, 2),
		oCh:  make(chan struct{}, 1),
		done: make(chan struct{}, 3),
	}
	h.hCh <- struct{}{}; h.hCh <- struct{}{}
	h.oCh <- struct{}{}
	return h
}

func (h *H2O) Hydrogen(releaseHydrogen func()) {
	<-h.hCh
	releaseHydrogen()
	h.done <- struct{}{}
	<-h.done; <-h.done; <-h.done // barrier: wait for 3
}

func (h *H2O) Oxygen(releaseOxygen func()) {
	<-h.oCh
	releaseOxygen()
	h.done <- struct{}{}
	<-h.done; <-h.done; <-h.done
	h.hCh <- struct{}{}; h.hCh <- struct{}{}
	h.oCh <- struct{}{}
}

### Rust

In [ ]:
use std::sync::{Arc, Mutex, Condvar};

struct H2O {
    h_count: Mutex<u8>,  // how many H have arrived this molecule
    o_ready: Mutex<bool>,
    cv: Condvar,
}

impl H2O {
    fn new() -> Arc<Self> {
        Arc::new(H2O {
            h_count: Mutex::new(0),
            o_ready: Mutex::new(false),
            cv: Condvar::new(),
        })
    }

    fn hydrogen(&self, release_hydrogen: impl Fn()) {
        let mut h = self.h_count.lock().unwrap();
        while *h >= 2 { h = self.cv.wait(h).unwrap(); }
        *h += 1;
        let ready = *h == 2 && *self.o_ready.lock().unwrap();
        drop(h);
        release_hydrogen();
        if ready { self.reset(); }
    }

    fn oxygen(&self, release_oxygen: impl Fn()) {
        let mut o = self.o_ready.lock().unwrap();
        while *o { o = self.cv.wait(o).unwrap(); }
        *o = true;
        let ready = *self.h_count.lock().unwrap() == 2;
        drop(o);
        release_oxygen();
        if ready { self.reset(); }
    }

    fn reset(&self) {
        *self.h_count.lock().unwrap() = 0;
        *self.o_ready.lock().unwrap() = false;
        self.cv.notify_all();
    }
}

## Concurrency Scenarios

1. **Two H arrive before O**: Both acquire `hSem` (capacity 2), hit the barrier, wait; O arrives, all three pass the barrier together.
2. **O arrives before both H**: O acquires `oSem`, waits at barrier; H1 and H2 arrive, all three rendezvous and proceed.
3. **Only one H arrives at start**: H1 acquires `hSem`, waits at barrier; H2 must arrive before O before anyone proceeds.
4. **Three molecules in sequence**: After oxygen releases the semaphores, the next HHO group can immediately begin acquiring — no starvation.
5. **Many H, few O**: Extra H threads block on `hSem` (max 2); once O completes a molecule and resets semaphores, the next two H are unblocked.
